In [ ]:
# Resolve public-release paths without requiring internal project documentation.
from pathlib import Path as _ReleasePath
import os as _release_os
_release_root = next((p for p in (_ReleasePath.cwd(), *_ReleasePath.cwd().parents)
                      if (p / ".shap_multiplicity_root").is_file()), None)
if _release_root is None:
    raise FileNotFoundError("Launch Jupyter within this repository.")
_release_os.chdir(_release_root)

import os, pickle, itertools, warnings
warnings.filterwarnings('ignore')

import numpy as np
from pairwise_aggregation import pairwise_distances, pairwise_exceedance
import matplotlib, matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines

try:
    import scienceplots
    plt.style.use(['science', 'no-latex'])
except ImportError:
    print('scienceplots not installed; using default style')

from plot_violin import (
    compute_rbo_distance, compute_jaccard_distance, compute_l2_distance,
    compute_rbo_baseline, compute_jaccard_baseline, compute_l2_baseline,
)

matplotlib.rcParams.update({
    'font.size':        13,
    'axes.labelsize':   14,
    'axes.titlesize':   14,
    'xtick.labelsize':  11,
    'ytick.labelsize':  12,
    'legend.fontsize':  11,
    'figure.titlesize': 15,
})

DATASETS = ['german', 'diabetes', 'acs', 'gmsc']
MODELS   = ['mlp', 'ftt']
METHODS  = ['random', 'kmeans', 'pred_stratified', 'cte']
METRICS  = ['l2', 'jac', 'rbo']

# Candidate bg_sizes per dataset (capped at trainset size; load_sv skips missing files)
CANDIDATE_BG_SIZES = {
    'german':   [25, 50, 100, 200, 400, 790],
    'diabetes': [25, 50, 100, 200, 313],
    'acs':      [25, 50, 100, 200, 400, 800, 1600],
    'gmsc':     [25, 50, 100, 200, 400, 800, 1600],
}

METHOD_LABELS = {'random': 'Random', 'kmeans': 'KMeans',
                 'pred_stratified': 'PredStrat', 'cte': 'CTE'}
METHOD_COLORS = {'random': '#4393C3', 'kmeans': '#D6604D',
                 'pred_stratified': '#56A65A', 'cte': '#9970AB'}
MODEL_LS      = {'mlp': '-',  'ftt': '--'}
MODEL_MARKERS = {'mlp': 'o',  'ftt': 's'}

METRIC_LABELS  = {'l2': 'L2 Distance', 'jac': 'Top-3 Jaccard', 'rbo': '1−RBO'}
DATASET_LABELS = {'german': 'German', 'diabetes': 'Diabetes', 'acs': 'ACS', 'gmsc': 'GMSC'}
N_FEATS        = {'german': 16, 'diabetes': 8, 'acs': 8, 'gmsc': 10, 'arrhythmia': 278}

JAC_K      = 3
N_BG_SEEDS = 5
# Reuse exactly the main-table K100 inputs; K100 was excluded from the sweep jobs.
K100_N_CHUNKS = {'german': 1, 'diabetes': 1, 'acs': 100, 'gmsc': 100}
PAIRS_10   = list(itertools.combinations(range(N_BG_SEEDS), 2))

RES_DIR   = 'results_tree'
BGRES_DIR = 'background_results'
TRUE_DIR  = 'results_true'
FIG_DIR   = 'figures_b/background_sweep'
os.makedirs(FIG_DIR, exist_ok=True)

print('Constants loaded.')

In [ ]:
# K100 reuses the same whole/chunk inputs as background_table.ipynb.
# Keep the alternate whole files on disk; do not fall back between result versions.
def load_sv(ds, model, method, bg_size, s_idx):
    if bg_size == 100:
        stem = f'{ds}_{model}_0_0_{method}_s{s_idx}'
        n_chunks = K100_N_CHUNKS[ds]
        paths = ([os.path.join(RES_DIR, f'{stem}_sv.pkl')] if n_chunks == 1 else
                 [os.path.join(RES_DIR, f'{stem}_c{c}_sv.pkl') for c in range(n_chunks)])
    else:
        paths = [os.path.join(BGRES_DIR,
                 f'{ds}_{model}_0_0_{method}_bg{bg_size}_s{s_idx}_sv.pkl')]
    if any(not os.path.exists(path) for path in paths):
        return None
    arrays = []
    for path in paths:
        with open(path, 'rb') as handle:
            obj = pickle.load(handle)
        arrays.append(np.asarray(obj.values if hasattr(obj, 'values') else obj['values']))
    return np.concatenate(arrays, axis=0)


def dist_fn(metric, v1, v2):
    if metric == 'l2':  return compute_l2_distance(v1, v2)
    if metric == 'jac': return compute_jaccard_distance(v1, v2, k=JAC_K)
    if metric == 'rbo': return compute_rbo_distance(v1, v2)


print('Helpers ready: K100 matches the main-table input policy.')


In [ ]:
# B inputs: raw (10, N) pair distances, flattened only for plotting.
# Retain the two axes separately for threshold counts; do not infer N from a pool.
sens_raw = {ds: {mo: {me: {mt: {} for mt in METRICS} for me in METHODS}
                 for mo in MODELS} for ds in DATASETS}
sens_pairs = {ds: {mo: {me: {mt: {} for mt in METRICS} for me in METHODS}
                   for mo in MODELS} for ds in DATASETS}
missing_groups = []
for ds in DATASETS:
    for model in MODELS:
        for method in METHODS:
            for bg_size in CANDIDATE_BG_SIZES[ds]:
                svs = [load_sv(ds, model, method, bg_size, s) for s in range(N_BG_SEEDS)]
                for mt in METRICS:
                    try:
                        distances = pairwise_distances(svs, lambda a, b: dist_fn(mt, a, b))
                        sens_pairs[ds][model][method][mt][bg_size] = distances
                        sens_raw[ds][model][method][mt][bg_size] = distances.reshape(-1)
                    except ValueError as error:
                        missing_groups.append(dict(dataset=ds, model=model, method=method,
                                                   bg_size=bg_size, metric=mt, reason=str(error)))
    print(f'{ds}: done')
print(f'Raw B arrays computed; {len(missing_groups)} unavailable metric/configuration groups.')


In [ ]:
# ── Baselines ─────────────────────────────────────────────────────────────────

baselines = {}
for ds in DATASETS:
    d = N_FEATS[ds]
    baselines[ds] = {
        'l2':  compute_l2_baseline(d=d, r=max(1, d // 3)),
        'jac': compute_jaccard_baseline(d=d, r=JAC_K),
        'rbo': compute_rbo_baseline(n_features=d),
    }
    print(f"{ds}: l2=[{baselines[ds]['l2']['min']:.4f},{baselines[ds]['l2']['max']:.4f}]  "
          f"jac=[{baselines[ds]['jac']['min']:.4f},{baselines[ds]['jac']['max']:.4f}]  "
          f"rbo=[{baselines[ds]['rbo']['min']:.4f},{baselines[ds]['rbo']['max']:.4f}]")

In [ ]:
# ── Grouped violin plots: 1 figure per (dataset, model) ──────────────────────
# Each figure: 1×3 subplots (one per metric)
# Each subplot: 4 grouped violins per bg_size (one per method, coloured)
# Red band = baseline; white line = median

W       = 0.16   # width of each violin
OFFSETS = np.array([-1.5, -0.5, 0.5, 1.5]) * W * (1 / 0.16) * 0.22  # ~[-0.33,-0.11,0.11,0.33]

for ds in DATASETS:
    bg_sizes_ds = CANDIDATE_BG_SIZES[ds]
    pos_base    = np.arange(len(bg_sizes_ds))   # integer x positions

    for model in MODELS:
        fig, axes = plt.subplots(1, 3, figsize=(14, 4))
        fig.suptitle(f'{DATASET_LABELS[ds]}  ·  {model.upper()}',
                     fontsize=14, fontweight='bold')

        for ax, mt in zip(axes, METRICS):
            for mi, method in enumerate(METHODS):
                vdata, vpos = [], []
                for k, bg_size in enumerate(bg_sizes_ds):
                    arr = sens_raw[ds][model][method][mt].get(bg_size)
                    if arr is None or len(arr) == 0:
                        continue
                    vdata.append(arr)
                    vpos.append(pos_base[k] + OFFSETS[mi])

                if not vdata:
                    continue

                parts = ax.violinplot(vdata, positions=vpos, widths=W,
                                      showmedians=True, showextrema=False)
                for pc in parts['bodies']:
                    pc.set_facecolor(METHOD_COLORS[method])
                    pc.set_edgecolor('none')
                    pc.set_alpha(0.75)
                parts['cmedians'].set_color('white')
                parts['cmedians'].set_linewidth(1.2)
                parts['cmedians'].set_zorder(5)

            # Baseline
            lo = baselines[ds][mt]['min']
            hi = baselines[ds][mt]['max']
            ax.axhspan(lo, hi, color='red', alpha=0.13, zorder=0)

            ax.set_xticks(pos_base)
            ax.set_xticklabels([str(sz) for sz in bg_sizes_ds],
                               rotation=45, ha='right', fontsize=10)
            ax.set_xlim(-0.6, len(bg_sizes_ds) - 0.4)
            ax.set_xlabel('Background Size', fontsize=13)
            ax.set_ylabel(METRIC_LABELS[mt], fontsize=13)
            ax.set_title(METRIC_LABELS[mt], fontsize=13)
            ax.tick_params(labelsize=11)

        # Legend
        method_handles = [mpatches.Patch(color=METHOD_COLORS[m], label=METHOD_LABELS[m])
                          for m in METHODS]
        bl_handle = mpatches.Patch(color='red', alpha=0.3, label='Baseline')
        axes[0].legend(handles=method_handles + [bl_handle],
                       fontsize=10, framealpha=0.85)

        fig.tight_layout()
        fname = os.path.join(FIG_DIR, f'background_sweep_violin_{ds}_{model}.pdf')
        fig.savefig(fname, bbox_inches='tight')
        plt.show(); plt.close(fig)
        print(f'Saved: {fname}')

In [ ]:
# ── % above baseline heatmaps: 1 figure per (dataset, model) ─────────────────
# Rows = methods, Cols = bg_size, Cell = % of (pair, instance) dist > baseline_min
# 1×3 subplots per figure (one per metric), matching the violin plot layout.

import seaborn as sns
import pandas as pd

aggregation_records = []
for ds in DATASETS:
    bg_sizes_ds = CANDIDATE_BG_SIZES[ds]

    for model in MODELS:
        fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
        fig.suptitle(f'{DATASET_LABELS[ds]}  ·  {model.upper()}  — % above baseline',
                     fontsize=14, fontweight='bold')

        for ax, mt in zip(axes, METRICS):
            bl = baselines[ds][mt]['min']
            mat = np.full((len(METHODS), len(bg_sizes_ds)), float('nan'))

            for mi, method in enumerate(METHODS):
                for bj, bg_size in enumerate(bg_sizes_ds):
                    arr = sens_pairs[ds][model][method][mt].get(bg_size)
                    if arr is not None and len(arr) > 0:
                        stats = pairwise_exceedance(arr, bl)
                        mat[mi, bj] = stats['percent']
                        aggregation_records.append(dict(dataset=ds, model=model, method=method,
                                                        bg_size=bg_size, metric=mt, **stats))

            df = pd.DataFrame(mat,
                              index=[METHOD_LABELS[m] for m in METHODS],
                              columns=[str(sz) for sz in bg_sizes_ds])

            annot = np.where(np.isnan(mat), '',
                             np.vectorize(lambda v: f'{v:.0f}')(mat))
            sns.heatmap(df, ax=ax, vmin=0, vmax=100, cmap='YlOrRd',
                        annot=annot, fmt='', annot_kws={'size': 9},
                        linewidths=0.3, linecolor='white',
                        cbar_kws={'label': '% comparisons > baseline', 'shrink': 0.8})

            ax.set_title(METRIC_LABELS[mt], fontsize=12)
            ax.set_xlabel('Background Size', fontsize=11)
            ax.set_ylabel('')
            ax.tick_params(axis='x', rotation=45, labelsize=9)
            ax.tick_params(axis='y', rotation=0, labelsize=10)

        fig.tight_layout()
        fname = os.path.join(FIG_DIR, f'background_sweep_heatmap_{ds}_{model}.pdf')
        fig.savefig(fname, bbox_inches='tight')
        plt.show(); plt.close(fig)
        print(f'Saved: {fname}')

pd.DataFrame(aggregation_records).to_csv(os.path.join(FIG_DIR, 'pair_exceedance_counts.csv'), index=False)
pd.DataFrame(missing_groups, columns=['dataset', 'model', 'method', 'bg_size', 'metric', 'reason']).to_csv(os.path.join(FIG_DIR, 'unavailable_groups.csv'), index=False)


In [ ]:
# (Old violin plot helpers — superseded by cells above)

In [ ]:
# (Old violin sensitivity plot — superseded by cells above)

## Historical cell — excluded from the B analysis

This preserved code depends on superseded variables. Use cells 1–6 for the current K≤1600 analysis. It is shown as text so Run All does not execute it.

```python
# ── Sensitivity: individual (1 per dataset × metric) ─────────────────────────

for ds in DATASETS:
    for mt in METRICS:
        nonempty = [(pos, sens[ds][sz][mt])
                    for pos, sz in zip(BG_POSITIONS, BG_SIZES)
                    if len(sens[ds][sz][mt])]
        if not nonempty:
            continue

        fig, ax = plt.subplots(figsize=(5, 4))
        styled_violin(ax, [d for _, d in nonempty],
                      [p for p, _ in nonempty], VIOLIN_COLOR)
        bl_patch = add_baseline_sweep(ax, mt, ds)
        style_ax(ax, mt)
        ax.legend(handles=[bl_patch], fontsize=12)

        fig.subplots_adjust(left=0.20, right=0.97, bottom=0.16, top=0.95)
        fname = os.path.join(FIG_DIR, f'background_sweep_{ds}_{mt}.pdf')
        fig.savefig(fname)
        plt.show(); plt.close(fig)
        print(f'Saved: {fname}')
```


## Historical cell — excluded from the B analysis

This preserved code depends on superseded variables. Use cells 1–6 for the current K≤1600 analysis. It is shown as text so Run All does not execute it.

```python
# ── True SHAP comparison: combined (1 per dataset, 3 subplots) ────────────────

for ds in DATASETS:
    if all(len(true_cmp[ds][sz][METRICS[0]]) == 0 for sz in BG_SIZES):
        print(f'{ds}: no true SHAP data, skip')
        continue

    fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=False)
                #  fontsize=13, fontweight='bold')

    bl_patch = None
    for ax, mt in zip(axes, METRICS):
        nonempty = [(pos, true_cmp[ds][sz][mt])
                    for pos, sz in zip(BG_POSITIONS, BG_SIZES)
                    if len(true_cmp[ds][sz][mt])]
        if nonempty:
            styled_violin(ax, [d for _, d in nonempty],
                          [p for p, _ in nonempty], VIOLIN_COLOR)
        bl_patch = add_baseline_sweep(ax, mt, ds)
        style_ax(ax, mt)

    axes[0].legend(handles=[bl_patch], fontsize=13)
    fig.tight_layout()
    fname = os.path.join(FIG_DIR, f'background_true_sweep_{ds}.pdf')
    fig.savefig(fname, bbox_inches='tight')
    plt.show(); plt.close(fig)
    print(f'Saved: {fname}')
```


## Historical cell — excluded from the B analysis

This preserved code depends on superseded variables. Use cells 1–6 for the current K≤1600 analysis. It is shown as text so Run All does not execute it.

```python
# ── True SHAP comparison: individual (1 per dataset × metric) ────────────────

for ds in DATASETS:
    for mt in METRICS:
        nonempty = [(pos, true_cmp[ds][sz][mt])
                    for pos, sz in zip(BG_POSITIONS, BG_SIZES)
                    if len(true_cmp[ds][sz][mt])]
        if not nonempty:
            continue

        fig, ax = plt.subplots(figsize=(5, 4))
        styled_violin(ax, [d for _, d in nonempty],
                      [p for p, _ in nonempty], VIOLIN_COLOR)
        bl_patch = add_baseline_sweep(ax, mt, ds)
        style_ax(ax, mt)
        ax.legend(handles=[bl_patch], fontsize=12)

        fig.subplots_adjust(left=0.20, right=0.97, bottom=0.16, top=0.95)
        fname = os.path.join(FIG_DIR, f'background_true_sweep_{ds}_{mt}.pdf')
        fig.savefig(fname)
        plt.show(); plt.close(fig)
        print(f'Saved: {fname}')
```
